# Set up

In [1]:
# 1. Hadoop configuration for Spark to access Hadoop binaries
import os
import duckdb

# Set HADOOP_HOME to the parent folder of 'bin'
os.environ['HADOOP_HOME'] = "C:/hadoop"
# Add the bin directory to your PATH
os.environ['PATH'] += os.pathsep + "C:/hadoop/bin"

# 2. Create a SparkSession with the DuckDB connector
from pyspark.sql import SparkSession
# Now run your session creation
duckdb_path = "C:\\Users\\jiaha\\Documents\\Universidad\\BDA\\BCN-Meteorologics\\trust_zone\\duckdb.jar"
spark = SparkSession.builder \
    .config("spark.jars", duckdb_path) \
    .getOrCreate()

# 3. Configuración de archivos
formatted_db_path = "C:\\Users\\jiaha\\Documents\\Universidad\\BDA\\BCN-Meteorologics\\formatted_zone.duckdb"
trusted_db_path = "C:\\Users\\jiaha\\Documents\\Universidad\\BDA\\BCN-Meteorologics\\trust_zone\\trusted_zone.duckdb"

# Análisis de Formatted Zone

In [2]:
import sys
sys.path.insert(0, "..")

from trust_zone.Analysis import analyze_formatted_zone, get_table_summary

# Ejecutar análisis completo de la formatted zone
analysis_report = analyze_formatted_zone(formatted_db_path)


📊 ANÁLISIS DE FORMATTED ZONE - 2026-04-11 16:54:12

Total de tablas encontradas: 3


────────────────────────────────────────────────────────────────────────────────
📋 TABLE: 2025_ACCIDENTS_GU_BCN
────────────────────────────────────────────────────────────────────────────────

🔹 Información Básica:
   ├─ Filas: 7,741
   └─ Columnas: 25

🔹 Esquema de Columnas:
   ├─ numero_expedient               (VARCHAR)
   ├─ codi_districte                 (INTEGER)
   ├─ nom_districte                  (VARCHAR)
   ├─ codi_barri                     (INTEGER)
   ├─ nom_barri                      (VARCHAR)
   ├─ codi_carrer                    (INTEGER)
   ├─ nom_carrer                     (VARCHAR)
   ├─ num_postal                     (VARCHAR)
   ├─ descripcio_dia_setmana         (VARCHAR)
   ├─ nk_any                         (INTEGER)
   ├─ mes_any                        (INTEGER)
   ├─ nom_mes                        (VARCHAR)
   ├─ dia_mes                        (INTEGER)
   ├─ hora_dia           

In [3]:
# Ejemplos de uso adicionales:

# 1. Obtener resumen de una tabla específica
# get_table_summary("2025_accidents_gu_bcn")

# 2. Exportar análisis a archivos CSV
# from Analysis import export_analysis_to_csv
# export_analysis_to_csv(output_dir="analysis_reports")

# Aux funtions
## 1. Módulo de Ingesta (Extract)
La función read_duckdb actúa como la interfaz de lectura. Se implementa una conexión mediante el driver JDBC de DuckDB, gestionando identificadores de tablas sensibles (como aquellos que inician con valores numéricos) mediante el uso de identificadores citados (quoted_table_name).

## 2. Lógica de Transformación (Transform)
### 2.1. Normalización Meteorológica (transform_meteo)
El objetivo es transformar un dataset de observaciones en formato "long" a un formato "wide" (tabular) para facilitar su consumo analítico.

Operación de Pivoteo: Se agrupan los datos por data_lectura y se pivotan los acrónimos de sensores (acronim).

Agregación: Se aplica el promedio (avg) sobre los valores para consolidar múltiples lecturas diarias en una única observación representativa de la ciudad.

Possible mejora: Estandarización Semántica: Se renombran los indicadores técnicos a términos de negocio: TM (Temperatura Media), HRM (Humedad Media) y PPT24H (Precipitación acumulada).

### 2.2. Agregación de Entidades de Personas (transform_persones)
Para evitar la duplicación de registros de accidentes por cada persona implicada, se realiza una pre-agregación a nivel de expediente:

Métricas Cuantitativas: Cálculo del total de implicados y edad media (redondeada a un decimal).

Segmentación Demográfica: Uso de lógica condicional (F.when) para contabilizar el volumen de hombres, mujeres y conductores por cada numero_expedient.

### 2.3. Consolidación y Enriquecimiento de Accidentes (transform_accidents)
Esta es la transformación nuclear del sistema, donde se aplican tres capas de procesamiento:

#### A. Limpieza y Tipado Estricto
Construcción de Timestamp: Se genera una clave temporal única mediante la concatenación de campos de año, mes, día y hora, utilizando el patrón flexible y-M-d-H para asegurar la compatibilidad con fechas no normalizadas.

Tratamiento de Nulos: Aplicación de la función coalesce para garantizar que las columnas de conteo (muertos y heridos) posean un valor por defecto de 0 en lugar de nulos.

Casting Geoespacial: Conversión explícita de coordenadas a tipo double para su posterior uso en sistemas GIS.

Deduplicación: Eliminación de registros redundantes basados en la clave primaria de negocio numero_expedient.

#### B. Enriquecimiento mediante Joins
Dimensión Personas: Se realiza un Left Join con la tabla agregada de personas para añadir contexto demográfico a cada accidente sin perder registros de la tabla principal.

Dimensión Ambiental: Se ejecuta un Left Join con los datos meteorológicos. La unión se realiza truncando el timestamp del accidente a nivel de fecha (F.to_date) para cruzarlo con la fecha de lectura meteorológica, permitiendo analizar el impacto del clima en la siniestralidad.

In [ ]:
from pyspark.sql import SparkSession, functions as F
# Load a table from DuckDB into a Spark DataFrame
def read_duckdb(table_name, path):
    formatted_path = path.replace("\\", "/")
    
    # Envolvemos el nombre de la tabla en comillas dobles 
    # para que DuckDB no se queje por el número "2025" al inicio.
    quoted_table_name = f'"{table_name}"'
    
    return spark.read \
        .format("jdbc") \
        .option("url", f"jdbc:duckdb:{formatted_path}") \
        .option("dbtable", quoted_table_name) \
        .option("driver", "org.duckdb.DuckDBDriver") \
        .load()
# Transformación de datos meteorológicos: pivotar meteorología, agregar personas, limpiar accidentes
def transform_meteo(df_meteo):
    # Pivotamos los acrónimos (TM, TX, TN, HRM, etc.) 
    # Promediamos valores por si hay varias lecturas en un día para la ciudad
    df_pivot = df_meteo.groupBy("data_lectura").pivot("acronim").agg(F.avg("valor"))
    
    # Mejora: Renombramos columnas para mayor claridad
    # df_pivot = df_pivot.withColumnRenamed("TM", "temp_media_diaria") \
    #                   .withColumnRenamed("HRM", "humedad_media_diaria") \
    #                   .withColumnRenamed("PPT24H", "precipitacion_24h")
    return df_pivot
# Agregación de datos de personas por accidente
def transform_persones(df_persones):
    return df_persones.groupBy("numero_expedient").agg(
        F.count("numero_expedient").alias("total_personas_implicadas"),
        F.round(F.avg("edat"), 1).alias("edad_media_implicados"),
        # Contamos cuántos hombres y mujeres
        F.count(F.when(F.col("descripcio_sexe") == "H", 1)).alias("num_hombres"),
        F.count(F.when(F.col("descripcio_sexe") == "D", 1)).alias("num_mujeres"),
        # Identificamos si hubo conductores o peatones
        F.count(F.when(F.col("descripcio_tipus_persona") == "Conductor", 1)).alias("num_conductores")
    )
# Limpieza y enriquecimiento de datos de accidentes
def transform_accidents(df_acc, df_pers_agg, df_meteo_wide):
    # 1. Limpieza de nulos y tipos
    df_clean = df_acc.select(
        "numero_expedient",
        "nom_districte",
        "nom_barri",
        "nom_carrer",
        
        # Parseo flexible de fecha y hora
        F.to_timestamp(
            F.concat_ws("-", "nk_any", "mes_any", "dia_mes", "hora_dia"), 
            "y-M-d-H"
        ).alias("fecha_hora_accidente"),
        
        F.coalesce(F.col("numero_morts").cast("int"), F.lit(0)).alias("total_muertos"),
        F.coalesce(F.col("numero_lesionats_greus").cast("int"), F.lit(0)).alias("total_heridos_graves"),
        F.coalesce(F.col("numero_lesionats_lleus").cast("int"), F.lit(0)).alias("total_heridos_leves"),
        
        F.col("longitud_wgs84").cast("double"),
        F.col("latitud_wgs84").cast("double")
    ).dropDuplicates(["numero_expedient"])

    # 2. Joins de Enriquecimiento
    
    # Unimos con los datos agregados de personas (Heridos/Muertos por tipo de persona)
    # Usamos "left" para no perder accidentes si no hay datos en la tabla de personas
    df_enriched = df_clean.join(df_pers_agg, on="numero_expedient", how="left")
    
    # Unimos con meteorología usando la fecha (truncamos la hora del accidente para cruzar con el día)
    df_final = df_enriched.join(
        df_meteo_wide, 
        F.to_date(df_enriched.fecha_hora_accidente) == df_meteo_wide.data_lectura, 
        "left"
    ).drop(df_meteo_wide.data_lectura) # Eliminamos la columna duplicada del join

    # El return debe ir AL FINAL de todas las transformaciones
    return df_final

In [ ]:
# Carga de datos de la Formatted Zone
df_acc_raw = read_duckdb("2025_ACCIDENTS_GU_BCN", formatted_db_path)
df_per_raw = read_duckdb("2025_ACCIDENTS_PERSONES_GU_BCN", formatted_db_path)
df_met_raw = read_duckdb("2025_METEOCAT_DETALL_ESTACIONS", formatted_db_path)

# Procesamiento
meteo_trusted = transform_meteo(df_met_raw)
persones_agg = transform_persones(df_per_raw)
accidents_final = transform_accidents(df_acc_raw, persones_agg, meteo_trusted)

# --- PASO CLAVE: PRE-CREACIÓN DE LA TABLA ---
nombre_tabla = "T_ACCIDENTS_ENRICHED"

print(f"🔧 Preparando catálogo en DuckDB para '{nombre_tabla}'...")
con = duckdb.connect(trusted_db_path)

# Creamos una definición de columnas dinámica basada en el DataFrame de Spark
# Usamos VARCHAR para todas de forma temporal; Spark luego aplicará el esquema real al sobrescribir
columnas = ", ".join([f'"{c}" VARCHAR' for c in accidents_final.columns])

# Forzamos la creación de la tabla vacía
con.execute(f'DROP TABLE IF EXISTS "{nombre_tabla}"')
con.execute(f'CREATE TABLE "{nombre_tabla}" ({columnas})')
con.close()
# --------------------------------------------

# 3. Escritura desde Spark (ahora encontrará la tabla y no fallará)
accidents_final.write \
    .format("jdbc") \
    .option("url", f"jdbc:duckdb:{trusted_db_path.replace('\\', '/')}") \
    .option("driver", "org.duckdb.DuckDBDriver") \
    .option("dbtable", f'"{nombre_tabla}"') \
    .mode("overwrite") \
    .save()
print(f"💾 Guardando datos enriquecidos mediante JDBC.")

🔧 Preparando catálogo en DuckDB para 'T_ACCIDENTS_ENRICHED'...
💾 Guardando datos enriquecidos mediante JDBC...
✅ Proceso completado para: T_ACCIDENTS_ENRICHED


In [14]:
analysis_report_trust = get_table_summary(trusted_db_path, nombre_tabla)


📊 RESUMEN: C:\Users\jiaha\Documents\Universidad\BDA\BCN-Meteorologics\trust_zone\trusted_zone.duckdb
────────────────────────────────────────────────────────────
Filas: 7,741 | Columnas: 30

Tipos de datos:
numero_expedient                     object
nom_districte                        object
nom_barri                            object
nom_carrer                           object
fecha_hora_accidente         datetime64[us]
total_muertos                         int32
total_heridos_graves                  int32
total_heridos_leves                   int32
longitud_wgs84                      float64
latitud_wgs84                       float64
total_personas_implicadas             int64
edad_media_implicados               float64
num_hombres                           int64
num_mujeres                           int64
num_conductores                       int64
DVM10                               float64
DVVX10                              float64
humedad_media_diaria                float64
